In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation as LDA
from sklearn.feature_extraction import text
from sklearn.decomposition import PCA


In [2]:
CORPUS = pd.read_csv("CORPUS.csv", sep="|")
LIB = pd.read_csv("LIB.csv", sep="|", index_col=0)

# LDA TOPIC (4)

In [3]:
##LDA Topic
# Filter CORPUS to nouns only and convert to doc strings
BAG = ['song_id']

DOCS = CORPUS[CORPUS.pos.str.match(r'^NNS?$')]\
    .groupby(BAG).term_str\
    .apply(lambda x: ' '.join(map(str, x)))\
    .to_frame()\
    .rename(columns={'term_str': 'doc_str'})

# Create count matrix
my_stop_words = list(text.ENGLISH_STOP_WORDS)
count_engine = CountVectorizer(max_df=0.9, min_df=10, stop_words=my_stop_words)
count_model = count_engine.fit_transform(DOCS.doc_str)
TERMS = count_engine.get_feature_names_out()

# Run LDA
n_topics = 20
max_iter = 100
n_top_terms = 9
TNAMES = [f"T{str(x).zfill(len(str(n_topics)))}" for x in range(n_topics)]

topic_engine = LDA(n_components=n_topics, max_iter=max_iter)
topic_model = topic_engine.fit_transform(count_model)


In [14]:
COUNT = pd.DataFrame(
    count_model.toarray(),
    index=DOCS.index,
    columns=TERMS
)
COUNT.to_csv("/Users/natalieseah/Desktop/TextasData/LDA_COUNT_MATRIX.csv", sep="|")

In [5]:
# Document-topic matrix
THETA = pd.DataFrame(topic_model, index=DOCS.index, columns=TNAMES)

# Topic-term matrix
PHI = pd.DataFrame(topic_engine.components_, columns=TERMS, index=TNAMES)

# Top 5 topics by average document weight
top_topics = THETA.mean().sort_values(ascending=False).head(5)
top_topics

T05    0.089326
T18    0.079166
T16    0.076433
T12    0.075232
T17    0.060360
dtype: float64

In [6]:
for topic in top_topics.index:
    top_words = PHI.loc[topic].sort_values(ascending=False).head(5).index.tolist()
    print(topic, top_words)

T05 ['life', 'world', 'time', 'somebody', 'day']
T18 ['yeah', 'baby', 'hey', 'pre', 'time']
T16 ['way', 'heart', 'time', 'music', 'trigger']
T12 ['love', 'man', 'time', 'look', 'hand']
T17 ['mind', 'things', 'people', 'im', 'woman']


# LDA THETA (4)

In [7]:
##LDA Theta
THETA = pd.DataFrame(topic_model, index=DOCS.index, columns=TNAMES)
THETA.columns.name = 'topic_id'
THETA

topic_id,T00,T01,T02,T03,T04,T05,T06,T07,T08,T09,T10,T11,T12,T13,T14,T15,T16,T17,T18,T19
song_id,,,,,,,,,,,,,,,,,,,,
0,0.000595,0.000595,0.000595,0.000595,0.036542,0.000595,0.000595,0.000595,0.000595,0.000595,0.000595,0.724910,0.081978,0.000595,0.000595,0.000595,0.000595,0.000595,0.147046,0.000595
1,0.000676,0.000676,0.000676,0.185570,0.000676,0.000676,0.000676,0.000676,0.000676,0.000676,0.000676,0.088553,0.000676,0.000676,0.039341,0.478136,0.000676,0.000676,0.198265,0.000676
2,0.000769,0.000769,0.000769,0.000769,0.000769,0.000769,0.000769,0.000769,0.000769,0.000769,0.000769,0.000769,0.000769,0.000769,0.000769,0.000769,0.000769,0.768638,0.217516,0.000769
3,0.035040,0.158930,0.622918,0.012495,0.000431,0.000431,0.090146,0.000431,0.000431,0.000431,0.047533,0.027333,0.000431,0.000431,0.000431,0.000431,0.000431,0.000431,0.000431,0.000431
4,0.001064,0.001064,0.001064,0.084203,0.001064,0.001064,0.022631,0.245920,0.001064,0.001064,0.001064,0.001064,0.001064,0.001064,0.263284,0.001064,0.001064,0.146042,0.223027,0.001064
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4110,0.001190,0.001190,0.001190,0.001190,0.001190,0.910468,0.001190,0.001190,0.001190,0.001190,0.001190,0.001190,0.068103,0.001190,0.001190,0.001190,0.001190,0.001190,0.001190,0.001190
4111,0.004167,0.004167,0.004167,0.004167,0.004167,0.252207,0.004167,0.004167,0.151701,0.004167,0.004167,0.004167,0.209788,0.088826,0.004167,0.004167,0.079359,0.159786,0.004167,0.004167
4112,0.004167,0.004167,0.004167,0.004167,0.087297,0.004167,0.004167,0.004167,0.004167,0.420833,0.088215,0.336988,0.004167,0.004167,0.004167,0.004167,0.004167,0.004167,0.004167,0.004167


# LDA PHI (4)

In [8]:
##LDA PHI
PHI = pd.DataFrame(topic_engine.components_, columns=TERMS, index=TNAMES)
PHI.index.name = 'topic_id'
PHI.columns.name = 'term_str'
PHI

term_str,accident,act,actin,action,adam,adrenaline,advantage,adventure,advice,affair,...,york,youd,youll,youre,youth,youtube,youve,yuh,yup,zone
topic_id,,,,,,,,,,,,,,,,,,,,,
T00,0.05,0.050000,0.050000,8.866246,0.050000,0.050000,0.050000,0.05,0.050000,0.050000,...,0.050000,0.050000,0.050000,2.078155,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000
T01,0.05,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.05,0.050000,0.050000,...,0.050000,0.050000,0.050000,116.414335,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000
T02,3.05,52.463831,0.050000,0.050000,0.050000,7.129150,0.050000,0.05,0.050000,13.432125,...,0.050000,0.050000,0.050000,70.717132,0.050000,1.090647,18.528592,53.406312,0.050000,0.050000
T03,0.05,15.185977,0.050000,0.050000,0.050000,0.050000,0.050000,0.05,21.470068,0.050000,...,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,19.693688,0.050000,0.050000
T04,0.05,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.05,0.050000,0.050000,...,0.050000,0.050000,0.050000,0.050000,1.090406,0.050000,0.050000,0.050000,0.050000,0.050000
T05,0.05,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.05,0.050000,0.050000,...,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000
T06,0.05,0.050000,12.799238,0.050000,0.050000,0.050000,1.155457,0.05,0.050000,0.050000,...,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000
T07,0.05,0.050000,0.050000,0.050000,0.050000,2.979270,0.050000,0.05,2.303315,0.050000,...,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000
T08,8.05,0.050000,0.050000,0.050000,0.050000,0.050000,0.050000,0.05,0.050000,0.050000,...,0.050000,11.742245,0.050000,88.678164,0.050000,0.050000,0.050000,0.050000,0.050000,35.637874


# LDA + PCA Visualization (4)

In [9]:
# TOPICS summary
TOPICS = PHI.stack().groupby('topic_id')\
    .apply(lambda x: ' '.join(x.sort_values(ascending=False).head(n_top_terms).reset_index().term_str))\
    .to_frame('top_terms')
TOPICS['doc_weight_sum'] = THETA.sum()
TOPICS['term_freq'] = PHI.sum(1) / PHI.sum(1).sum()
TOPICS.sort_values('doc_weight_sum', ascending=False)

,top_terms,doc_weight_sum,term_freq
topic_id,,,
T05,life world time somebody day song reason oh girl,366.235050,0.066862
T18,yeah baby hey pre time day rules lights rihanna,324.578645,0.090738
T16,way heart time music trigger day babe words ri...,313.376235,0.068518
T12,love man time look hand memories wait need day,308.449462,0.073973
T17,mind things people im woman time friends god way,247.477358,0.054186
T15,eyes beyonc lights dream hell feels stars dadd...,244.029971,0.057339
T07,ooh thing rain tears woah gon lips time water,218.741805,0.049885
T09,home christmas jump arms year city california ...,191.562689,0.042418
T02,night come baby halo dreams mama da let day,190.589119,0.052474


In [15]:
##LDA + PCA Viz

# Find dominant artist per topic
THETA_LIB = THETA.join(LIB[['Artist']])
DOMINANT_ARTIST = THETA_LIB.groupby('Artist')[TNAMES].mean().idxmax(axis=0).to_frame('dominant_artist')

pca_lda = PCA(n_components=2)
THETA_PCA = pd.DataFrame(
    pca_lda.fit_transform(THETA.T),
    index=TNAMES,
    columns=['PC0', 'PC1']
)

THETA_PCA['top_terms'] = TOPICS['top_terms']
THETA_PCA['mean_weight'] = THETA.mean()
THETA_PCA['topic_id'] = THETA_PCA.index
THETA_PCA['dominant_artist'] = DOMINANT_ARTIST['dominant_artist']

import plotly.express as px

fig = px.scatter(
    THETA_PCA,
    x='PC0',
    y='PC1',
    size='mean_weight',
    color='dominant_artist',
    text='topic_id',
    hover_name='topic_id',
    hover_data={'top_terms': True, 'mean_weight': ':.4f', 'PC0': False, 'PC1': False},
    size_max=40,
    title='LDA Topics in PCA Space',
    height=800
)

fig.update_traces(textposition='top center')
fig.show()

In [16]:
#Another one more zoomed into the cluster

fig2 = px.scatter(
    THETA_PCA,
    x='PC0',
    y='PC1',
    size='mean_weight',
    color='dominant_artist',
    text='topic_id',
    hover_name='topic_id',
    hover_data={'top_terms': True, 'mean_weight': ':.4f', 'PC0': False, 'PC1': False},
    size_max=60,
    title='LDA Topics in PCA Space (zoomed in)',
    height=800
)

fig2.update_traces(textposition='top center')
fig2.update_xaxes(range=[-2, 2])
fig2.update_yaxes(range=[-2, 2])
fig2.show()

In [12]:
THETA.to_csv('/Users/natalieseah/Desktop/TextasData/LDA_THETA.csv', sep='|')
PHI.to_csv('/Users/natalieseah/Desktop/TextasData/LDA_PHI.csv', sep='|')
TOPICS.to_csv('/Users/natalieseah/Desktop/TextasData/LDA_TOPICS.csv', sep='|')